In [1]:
import random

# Define possible nucleotides
nucleotides = ['A', 'C', 'G', 'T']

# Define mutation rate per site (chance that a site will mutate each generation)
mutation_rate = 0.1

# Define the number of generations to simulate
num_generations = 10

# Initialize a random starting sequence (finite length)
sequence_length = 20
sequence = [random.choice(nucleotides) for _ in range(sequence_length)]

def mutate_site(nucleotide):
    """Mutate a given nucleotide to one of the other three nucleotides."""
    possible_mutations = [nuc for nuc in nucleotides if nuc != nucleotide]
    return random.choice(possible_mutations)

# Simulate sequence evolution over generations
print("Generation 0:", ''.join(sequence))

for generation in range(1, num_generations + 1):
    new_sequence = []
    for site in sequence:
        # Mutate with a probability equal to mutation_rate
        if random.random() < mutation_rate:
            new_sequence.append(mutate_site(site))
        else:
            new_sequence.append(site)
    sequence = new_sequence  # Update sequence for the next generation
    print(f"Generation {generation}: {''.join(sequence)}")


Generation 0: ACATATGCTGTTGGTTACTC
Generation 1: ACATATGCTGTTGGTTACGC
Generation 2: ACATACGCTGTTGGTTAGGC
Generation 3: ACGTACGCTGTTTGTTAGGC
Generation 4: ACGTACGATGTTTGTGATGC
Generation 5: ACGTACGATGTTTGTTATGC
Generation 6: ACGTACGATGTATGTTATGC
Generation 7: ACGTACGATGTATGTTATGC
Generation 8: CCGTACGATGTATGTTATGA
Generation 9: GCGTTCGTTGTATGTTATGA
Generation 10: GCGGTCGTTGTATATTTTGA


In [14]:
import numpy as np

# Dummy functions for calculating likelihood, generating trees, and models
def calculate_likelihood(tree, theta, model):
    # Placeholder likelihood function
    # Acknowledge the parameters to avoid compile errors
    _ = tree
    _ = theta
    _ = model
    return np.random.uniform(0, 1)  # returns a random likelihood for demonstration

def generate_random_tree(num_leaves):
    # Placeholder for a random tree structure generation
    return {"tree_structure": f"Tree with {num_leaves} leaves"}

def sample_theta():
    # Placeholder for sampling new parameter theta
    return np.random.uniform(0, 1)

def sample_model():
    # Placeholder for sampling new model configuration
    return {"model_param": np.random.uniform(0, 1)}

def phylo_tree_search(D, theta_0, M_lambda_0, n_iter, p1, p2, m, mh):
    # Initialize
    T = generate_random_tree(m)  # Random initial tree with m leaves
    theta = theta_0
    M_lambda = M_lambda_0
    L = calculate_likelihood(T, theta, M_lambda)

    # Track best results
    L_best = L
    T_best = T
    theta_best = theta
    M_lambda_best = M_lambda

    # Iterative search
    for i in range(n_iter):
        # Propose new theta and model
        if np.random.uniform(0, 1) <= p1:
            theta_new = sample_theta()
            M_lambda_new = M_lambda
            rho_theta = min(1, calculate_likelihood(T, theta_new, M_lambda) / L)
            if np.random.uniform(0, 1) <= rho_theta:
                theta = theta_new
                M_lambda = M_lambda_new
        elif np.random.uniform(0, 1) <= (p1 + p2):
            M_lambda_new = sample_model()
            theta_new = theta
            rho_lambda = min(1, calculate_likelihood(T, theta, M_lambda_new) / L)
            if np.random.uniform(0, 1) <= rho_lambda:
                M_lambda = M_lambda_new
        else:
            # Propose a new tree
            T_new = generate_random_tree(m)
            r_T = min(1, calculate_likelihood(T_new, theta, M_lambda) / L)
            if np.random.uniform(0, 1) <= r_T:
                T = T_new

        # Calculate likelihood with updated values
        L = calculate_likelihood(T, theta, M_lambda)

        # Update best results if likelihood improves
        if L > L_best:
            L_best = L
            T_best = T
            theta_best = theta
            M_lambda_best = M_lambda

    # Return the best tree and parameters
    return L_best, T_best, theta_best, M_lambda_best

# Parameters for the search
D = None  # Placeholder for data
theta_0 = 0.5  # Initial theta
M_lambda_0 = {"model_param": 0.5}  # Initial model parameters
n_iter = 100  # Number of iterations
p1, p2 = 0.3, 0.3  # Probabilities for parameter/model updates
m = 5  # Number of leaves in the tree
mh = 0.1  # Metropolis-Hastings threshold (for simplicity, not directly used here)

# Run the phylogenetic tree search
best_likelihood, best_tree, best_theta, best_model = phylo_tree_search(D, theta_0, M_lambda_0, n_iter, p1, p2, m, mh)

print("Best Likelihood:", best_likelihood)
print("Best Tree:", best_tree)
print("Best Theta:", best_theta)
print("Best Model:", best_model)


Best Likelihood: 0.9951065965093829
Best Tree: {'tree_structure': 'Tree with 5 leaves'}
Best Theta: 0.6541003228041742
Best Model: {'model_param': 0.5102863840389731}


In [5]:
from scipy.stats import binom
# sequecing error
alt = 2
coverage = 20
sequencingError_probCutoff = 0.0001
seqQualityCutoff = 30
seqErrorRate = 10 ** -(seqQualityCutoff/10)
prob = 1 - binom.cdf(alt, coverage, seqErrorRate)
passed = prob < sequencingError_probCutoff
print(passed) 

True


In [3]:
import scipy.stats as stats

# Sample data: positions supporting the alternate and reference alleles
positions_alt = [10, 15, 18, 20, 22]  # Positions supporting the alternate allele
positions_ref = [2, 5, 6, 8, 11]      # Positions supporting the reference allele

# Perform Mann-Whitney U test
u_statistic, p_value = stats.mannwhitneyu(positions_alt, positions_ref, alternative='two-sided')

# Output results
print(f"U statistic: {u_statistic}")
print(f"p-value: {p_value}")

# Interpretation
if p_value < 0.05:
    print("There is a statistically significant difference in positions (p < 0.05).")
else:
    print("No significant difference in positions (p >= 0.05).")


U statistic: 24.0
p-value: 0.015873015873015872
There is a statistically significant difference in positions (p < 0.05).


In [ ]:
import scipy.stats as stats
'''
def integrate_baseQuality_onesite(oneSiteBaseQua):
    # Example: 32||||20|;27:32||||20|;32:37|||||
    basicBase = ["A", "T", "C", "G", "D", "I"]
    baseQuaDic = {base: [] for base in basicBase}
    cellsBaseQuaList = oneSiteBaseQua.split(";")
    for cell in cellsBaseQuaList:
        cellBaseList = cell.split("|")
        for index_base, cellBase in enumerate(cellBaseList):
            if cellBase:
                base = basicBase[index_base]
                baseQuaDic[base].append(cellBase)
    integrate_baseQuality = "|".join(":".join(baseQuaDic[base]) for base in basicBase)
    return integrate_baseQuality

def get_ref_alt_baseQuality_onesite(oneSiteBaseQua, refBase):
    # Example: 32:27:32:32:37||||20:20|
    basicBase = ["A", "T", "C", "G", "D", "I"]
    altBase = [base for base in basicBase if base != refBase]
    baseQuainfoList = oneSiteBaseQua.split("|")
    refBaseQua = [int(i) for i in baseQuainfoList[basicBase.index(refBase)].split(":") if i]
    altBaseQua = [int(i) for base in altBase if baseQuainfoList[basicBase.index(base)] != "" for i in baseQuainfoList[basicBase.index(base)].split(":") if i]
    return refBaseQua, altBaseQua
'''
def integrate_baseQuality_onesite(oneSiteBaseQua):
    # Define basic base labels
    basicBase = ["A", "T", "C", "G", "D", "I"]
    baseQuaDic = {base: [] for base in basicBase}
    
    # Split the input by semicolons to process each cell
    cellsBaseQuaList = oneSiteBaseQua.split(";")
    for cell in cellsBaseQuaList:
        cellBaseList = cell.split("|")
        for index_base, cellBase in enumerate(cellBaseList):
            if cellBase:
                # Map base to quality score and store it in the dictionary
                base = basicBase[index_base]
                baseQuaDic[base].append(cellBase)
    
    # Join each base's list of qualities, handling cases where some may be empty
    integrate_baseQuality = "|".join(
        ":".join(baseQuaDic[base]) if baseQuaDic[base] else "" for base in basicBase
    )
    return integrate_baseQuality


def get_ref_alt_baseQuality_onesite(oneSiteBaseQua, refBase):
    # Define basic base labels
    basicBase = ["A", "T", "C", "G", "D", "I"]
    
    # Check if refBase is valid
    if refBase not in basicBase:
        raise ValueError(f"Invalid reference base '{refBase}'")

    altBase = [base for base in basicBase if base != refBase]
    baseQuainfoList = oneSiteBaseQua.split("|")
    
    # Get quality values for the reference base
    try:
        refBaseQua = [
            int(i) for i in baseQuainfoList[basicBase.index(refBase)].split(":") if i
        ]
    except (IndexError, ValueError):
        refBaseQua = []  # Handle missing or malformed reference base quality data

    # Get quality values for the alternative bases
    altBaseQua = []
    for base in altBase:
        base_index = basicBase.index(base)
        if base_index < len(baseQuainfoList) and baseQuainfoList[base_index]:
            altBaseQua.extend(int(i) for i in baseQuainfoList[base_index].split(":") if i)
    
    return refBaseQua, altBaseQua


integrate_baseQuality = integrate_baseQuality_onesite("32||||20|;27:32||||20|;32:37|||||;32:37||33:44:55|||")
ref, alt = get_ref_alt_baseQuality_onesite(integrate_baseQuality, "A")
print(integrate_baseQuality)
print(ref)
print(alt)
u_statistic, p_value = stats.mannwhitneyu(alt, ref, alternative='two-sided')
print(f"U statistic: {u_statistic}")
print(f"p-value: {p_value}")
# Interpretation
if p_value < 0.05:
    print("There is a statistically significant difference in positions (p < 0.05).")
else:
    print("No significant difference in positions (p >= 0.05).")


32:27:32:32:37:32:37||33:44:55||20:20|
[32, 27, 32, 32, 37, 32, 37]
[33, 44, 55, 20, 20]
U statistic: 19.0
p-value: 0.8682217356878595
No significant difference in positions (p >= 0.05).
